<div dir="rtl">

# فحص مجموعة بيانات شبكة الاتصالات خطوة بخطوة
### Telecom Network Data — Exploratory Data Analysis & Validation

**إعداد:** م. زيد العنزي — قسم علوم الذكاء الاصطناعي، جامعة المصطفى

**أهداف الدفتر:**
1. رفع البيانات وقراءتها في Google Colab.
2. فحص جودة البيانات (القيم المفقودة، التكرار، الانتظام الزمني، القيم الشاذة).
3. التحليل الاستكشافي والإحصائي.
4. اكتشاف أن المتغير الهدف مُولَّد بقاعدة حتمية باستخدام شجرة قرار.
5. استنتاج دروس حول **عدم توازن الفئات** و**التحذير من الدقة 100%**.

> **ملاحظة:** عناوين الرسوم باللغة الإنجليزية لأن مكتبة matplotlib لا تعرض الحروف العربية بشكل صحيح افتراضيًا.
</div>

<div dir="rtl">

## الخطوة 1: استيراد المكتبات
</div>

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (classification_report, ConfusionMatrixDisplay,
                             accuracy_score, f1_score)

sns.set_theme(style='whitegrid')
pd.set_option('display.width', 120)
RANDOM_STATE = 42
print('تم تحميل المكتبات بنجاح')

<div dir="rtl">

## الخطوة 2: رفع ملف البيانات
عند تشغيل الخلية ستظهر نافذة لاختيار الملف `Telecom_Network_Data.csv` من جهازك.
</div>

In [ ]:
FILE_NAME = 'Telecom_Network_Data.csv'

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not os.path.exists(FILE_NAME):
    uploaded = files.upload()
    FILE_NAME = next(iter(uploaded))

print('الملف المستخدم:', FILE_NAME)

# بديل: القراءة من Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# FILE_NAME = '/content/drive/MyDrive/datasets/Telecom_Network_Data.csv'

<div dir="rtl">

## الخطوة 3: قراءة البيانات والفحص الأولي
</div>

In [ ]:
df = pd.read_csv(FILE_NAME, parse_dates=['timestamp'])
print('الأبعاد (صفوف، أعمدة):', df.shape)
df.info()
df.head()

<div dir="rtl">

## الخطوة 4: فحص جودة البيانات
</div>

In [ ]:
print('--- القيم المفقودة ---')
print(df.isnull().sum())

print('\n--- التكرار ---')
print('صفوف مكررة بالكامل:', df.duplicated().sum())
print('تكرار (البرج، الوقت):', df.duplicated(['timestamp', 'tower_id']).sum())

print('\n--- المدى الزمني ---')
print('من', df['timestamp'].min(), 'إلى', df['timestamp'].max())

print('\n--- عدد القراءات لكل برج ---')
print(df['tower_id'].value_counts().sort_index())

print('\n--- انتظام الفواصل الزمنية لكل برج ---')
for tower, g in df.groupby('tower_id'):
    gaps = g['timestamp'].sort_values().diff().dropna().value_counts()
    print(f'Tower {tower}:', dict(gaps))

<div dir="rtl">

## الخطوة 5: الإحصاءات الوصفية وتوازن الفئات
</div>

In [ ]:
NUM_COLS = ['users_connected', 'download_speed', 'upload_speed', 'latency']
display(df[NUM_COLS].describe().T.round(2))

print('توزيع حالات الطقس:')
print(df['weather'].value_counts())

print('\nتوزيع المتغير الهدف congestion:')
print(df['congestion'].value_counts())
print((df['congestion'].value_counts(normalize=True) * 100).round(2).astype(str) + ' %')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(x='congestion', data=df, ax=axes[0])
axes[0].set_title('Class Balance: congestion')
sns.countplot(x='weather', data=df, order=df['weather'].value_counts().index, ax=axes[1])
axes[1].set_title('Weather Distribution')
plt.tight_layout(); plt.show()

<div dir="rtl">

**ملاحظة:** نسبة الازدحام نحو **5.6%** فقط، أي أن البيانات **غير متوازنة**. نموذج يتنبأ دائمًا بـ 0 سيحقق دقة 94% دون أن يتعلم شيئًا.
</div>

<div dir="rtl">

## الخطوة 6: كشف القيم الشاذة والقيم غير المنطقية
</div>

In [ ]:
for col in NUM_COLS:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    n_out = ((df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)).sum()
    print(f'{col:17s} | شاذة (IQR): {n_out:4d} | سالبة: {(df[col] < 0).sum()} | صفرية: {(df[col] == 0).sum()}')

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, col in zip(axes, NUM_COLS):
    sns.boxplot(y=df[col], ax=ax); ax.set_title(col)
plt.tight_layout(); plt.show()

# فحص منطقي: سرعة الرفع أكبر من سرعة التنزيل (نادر في الشبكات الخلوية)
bad = (df['upload_speed'] > df['download_speed'])
print(f'\nصفوف سرعة الرفع فيها > سرعة التنزيل: {bad.sum()} ({bad.mean()*100:.1f}%)')

<div dir="rtl">

## الخطوة 7: فحص شكل التوزيعات
نختبر هل التوزيعات منتظمة (Uniform) باستخدام اختبار Kolmogorov-Smirnov. إذا كانت قيمة p > 0.05 فلا نرفض فرضية الانتظام، وهذا مؤشر على بيانات **اصطناعية**.
</div>

In [ ]:
df[NUM_COLS].hist(bins=30, figsize=(12, 7))
plt.suptitle('Feature Distributions'); plt.tight_layout(); plt.show()

rows = []
for col in NUM_COLS:
    x = df[col]
    scaled = (x - x.min()) / (x.max() - x.min())
    rows.append({'feature': col,
                 'skewness': round(x.skew(), 3),
                 'kurtosis': round(x.kurt(), 3),
                 'KS_uniform_p': round(stats.kstest(scaled, 'uniform').pvalue, 3)})
pd.DataFrame(rows)

<div dir="rtl">

**التفسير:** الالتواء قريب من الصفر، والتفرطح قريب من −1.2 (وهي القيمة النظرية للتوزيع المنتظم)، وقيم p كلها أكبر من 0.05. النتيجة: القيم مولَّدة عشوائيًا بتوزيع منتظم.
</div>

<div dir="rtl">

## الخطوة 8: مصفوفة الارتباط
</div>

In [ ]:
corr = df[NUM_COLS + ['congestion']].corr()
plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix'); plt.show()

<div dir="rtl">

**التفسير:** الارتباط بين الخصائص شبه معدوم (مثلًا لا علاقة بين عدد المستخدمين وزمن الاستجابة)، وهذا غير واقعي. أقوى ارتباط مع الهدف هو `users_connected` ثم `latency`.
</div>

<div dir="rtl">

## الخطوة 9: الازدحام حسب المتغيرات
</div>

In [ ]:
print('متوسط الخصائص حسب حالة الازدحام:')
display(df.groupby('congestion')[NUM_COLS].mean().round(2))

ct = pd.crosstab(df['weather'], df['congestion'])
print('نسبة الازدحام حسب الطقس:')
display(pd.crosstab(df['weather'], df['congestion'], normalize='index').round(3))
chi2, p, dof, _ = stats.chi2_contingency(ct)
print(f'اختبار مربع كاي: chi2={chi2:.2f}, p={p:.4f}')

df['hour'] = df['timestamp'].dt.hour
df.groupby('hour')['congestion'].mean().plot(kind='bar', figsize=(12, 3.5))
plt.title('Congestion Rate by Hour'); plt.ylabel('rate'); plt.show()

<div dir="rtl">

**تفسير اختبار مربع كاي:** قيمة p ≈ 0.058 أكبر من 0.05، أي أن الفرق في نسبة الازدحام بين حالات الطقس **غير دال إحصائيًا**.

**سؤال للطلاب:** هل يوجد نمط ذروة يومي واضح (صباحي/مسائي) كما نتوقع في شبكة حقيقية؟
</div>

<div dir="rtl">

## الخطوة 10: الرسم الحاسم — المستخدمون مقابل زمن الاستجابة
</div>

In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(data=df, x='users_connected', y='latency', hue='congestion',
                palette={0: 'lightgray', 1: 'crimson'}, s=15)
plt.axvline(800, ls='--', c='k'); plt.axhline(150, ls='--', c='k')
plt.title('Users vs Latency (dashed lines: 800 users, 150 ms)')
plt.show()

<div dir="rtl">

جميع نقاط الازدحام (الأحمر) تقع في الزاوية العليا اليمنى فقط. هذا يوحي بوجود **قاعدة حتمية**. لنتحقق منها بنموذج تعلم آلي.
</div>

<div dir="rtl">

## الخطوة 11: تجهيز البيانات وتقسيمها
</div>

In [ ]:
X = pd.get_dummies(df[['tower_id', 'hour'] + NUM_COLS + ['weather']],
                   columns=['weather'], drop_first=True)
y = df['congestion']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print('تدريب:', X_train.shape, '| اختبار:', X_test.shape)
print('نسبة الازدحام في التدريب والاختبار:', round(y_train.mean(), 3), round(y_test.mean(), 3))

<div dir="rtl">

## الخطوة 12: النموذج المرجعي (Baseline)
</div>

In [ ]:
dummy = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
y_dummy = dummy.predict(X_test)
print('Accuracy :', round(accuracy_score(y_test, y_dummy), 3))
print('F1 (class 1):', round(f1_score(y_test, y_dummy, zero_division=0), 3))

<div dir="rtl">

دقة مرتفعة جدًا مع F1 = 0، وهذا هو الدليل العملي على أن **Accuracy مضللة** مع البيانات غير المتوازنة.
</div>

<div dir="rtl">

## الخطوة 13: شجرة قرار بعمق 2 لكشف القاعدة
</div>

In [ ]:
tree = DecisionTreeClassifier(max_depth=2, class_weight='balanced',
                              random_state=RANDOM_STATE).fit(X_train, y_train)
y_pred = tree.predict(X_test)
print(classification_report(y_test, y_pred, digits=3))

print(export_text(tree, feature_names=list(X.columns)))

plt.figure(figsize=(12, 5))
plot_tree(tree, feature_names=X.columns, class_names=['0', '1'], filled=True)
plt.show()

ConfusionMatrixDisplay.from_predictions(y_test, y_pred); plt.show()

<div dir="rtl">

**لاحظ:** العتبات التي تعلمتها الشجرة (≈ 802.5 مستخدم و ≈ 149.7 ms) قريبة جدًا من القيم المستديرة 800 و150. الفرق سببه أن الشجرة تضع العتبة في منتصف المسافة بين أقرب قيمتين في بيانات التدريب.

## الخطوة 14: التحقق من القاعدة على كامل البيانات
</div>

In [ ]:
rule = ((df['users_connected'] > 800) & (df['latency'] > 150)).astype(int)
print('نسبة التطابق مع القاعدة:', (rule == df['congestion']).mean())
pd.crosstab(rule, df['congestion'], rownames=['rule'], colnames=['congestion'])

<div dir="rtl">

**النتيجة:** تطابق 100%. المتغير الهدف مولَّد بالقاعدة:

$$congestion = 1 \iff users\_connected > 800 \;\wedge\; latency > 150$$
</div>

<div dir="rtl">

## الخطوة 15: الغابة العشوائية والتحقق المتقاطع وأهمية الخصائص
</div>

In [ ]:
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced',
                            random_state=RANDOM_STATE)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scores = cross_val_score(rf, X, y, cv=cv, scoring='f1')
print(f'F1 (5-fold): {scores.mean():.3f} ± {scores.std():.3f}')

rf.fit(X_train, y_train)
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values()
imp.plot(kind='barh', figsize=(8, 5)); plt.title('Random Forest Feature Importance'); plt.show()

<div dir="rtl">

## الخطوة 16: تجربة الحذف (Ablation)
ماذا يحدث لو حذفنا الخاصيتين اللتين تعتمد عليهما القاعدة؟
</div>

In [ ]:
X_ab = X.drop(columns=['users_connected', 'latency'])
scores_ab = cross_val_score(rf, X_ab, y, cv=cv, scoring='f1')
print(f'F1 بدون users_connected و latency: {scores_ab.mean():.3f} ± {scores_ab.std():.3f}')

<div dir="rtl">

ينهار الأداء تقريبًا إلى الصفر، مما يؤكد أن باقي الخصائص (السرعات، الطقس، الساعة، البرج) **لا تحمل أي معلومة** عن الازدحام في هذه البيانات.
</div>

<div dir="rtl">

## الخلاصة

| البند | النتيجة |
|---|---|
| القيم المفقودة والتكرار | لا يوجد |
| الانتظام الزمني | منتظم، كل ساعة، 5 أبراج × 721 قراءة |
| القيم الشاذة (IQR) | لا يوجد |
| توازن الفئات | غير متوازن (≈ 5.6% ازدحام) |
| التوزيعات | منتظمة (Uniform) ← بيانات اصطناعية |
| الارتباطات بين الخصائص | شبه معدومة ← غير واقعي |
| خلل منطقي | ≈ 22% من الصفوف سرعة الرفع فيها أكبر من التنزيل |
| المتغير الهدف | قاعدة حتمية: users > 800 و latency > 150 |

**الدروس المستفادة:**
1. فحص البيانات يسبق بناء النموذج دائمًا.
2. الدقة (Accuracy) مضللة مع الفئات غير المتوازنة؛ استخدم F1 وRecall وPR-AUC.
3. الأداء شبه الكامل **علامة تحذير** تستدعي البحث عن قاعدة مخفية أو تسرّب بيانات.
4. النماذج القابلة للتفسير (شجرة القرار) أداة ممتازة لفهم البيانات.
5. هذه البيانات صالحة للتعليم، وغير صالحة لبحث علمي منشور.

## تمارين للطلاب
1. أضف ضوضاء إلى التسمية (اقلب 5% من القيم عشوائيًا) وأعد الخطوات 13–15. ماذا تلاحظ؟
2. جرّب `LogisticRegression`. لماذا يكون أداؤه أقل من شجرة القرار هنا؟ (تلميح: القاعدة عبارة عن تقاطع شرطين)
3. أنشئ خاصية جديدة `users_x_latency = users_connected * latency` وقارن الأداء.
4. استخدم `precision_recall_curve` وارسم منحنى PR للغابة العشوائية.
5. اقترح طريقة لتوليد بيانات أكثر واقعية (نمط ذروة يومي، علاقة بين المستخدمين وزمن الاستجابة).
</div>